# Schema Definition

### The raw CSV file that we have with ourselves is a result of web scraping where the same restaurants
### have been listed multiple times across different restaurant listings wih varying rating and count of votes 
### depending upon the time when it was scraped.
### So, our first line of action is to normalize this raw dataset to a form that is fit for further analysis and 
### modelling.
###
### The design of the Schema is as follows - 
###     1. restaurant (rest) : contains only the restaurant grain - (name, address), where each row has unique restaurant details along with their latest count of votes, votes = max of votes across different snapshots that act as a proxy for the latest accumulated votes, and the mean of their ratings, ratings are near-constant within a restaurant.
###     2. restaurant_listings : contains all the different categories in which each restaurant is listed in i.e. the listings grain - (name, address, listing). Will have features - name, address and listings where the relationship will be one-to-many.
###     3. restaurant_cuisines : contains all the different cusisines that each of the restaurants have to offer i.e. the cuisines grain - (name, address, cuisine). Will have features - name, address and cuisines where the relationship will be many-to-many.
###     4. restaurant_reviews(text): will have the text reviews. Here the text reviews are deferred because they do not fall under the scope of our current analysis, hence scoped-out.
###
### votes/ratings have been placed inside the restaurant table because they are properties of restaurant stored only once in the restaurant table for each restaurant. The raw file copied these values across rows rather than measuring them separately per snapshot, so they are a single restaurant-level fact stored once — which also prevents the same value drifting out of sync across tables.

## Stage 1 - Ingest

In [1]:
import pandas as pd
core = pd.read_csv('zomato.csv')
# dropping columns - reviews_list and menu_item as they are heavy text blobs and out of scope as per Schema
core = core.drop(columns=['reviews_list', 'menu_item'])
print('columns: ', core.columns)
print('count of rows: ', len(core))

columns:  Index(['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'listed_in(type)', 'listed_in(city)'],
      dtype='str')
count of rows:  51717


## Stage 2 - Type & Parse

In [2]:
import numpy as np
core['is_new'] = (core['rate'].astype(str).str.strip() == 'NEW').astype(int)
core['rate'] = pd.to_numeric(core['rate'].astype(str).str.split('/').str[0].str.strip().replace(['NEW', '-', 'nan', '', 'None'], np.nan), errors='coerce') 
cc = 'approx_cost(for two people)'
core['cost'] = pd.to_numeric(core[cc].astype(str).str.replace(',', '', regex=False), errors='coerce')
for c in ['online_order', 'book_table']:
    core[c] = core[c].astype(str).str.strip().map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

core['n_cuisines'] = core['cuisines'].str.count(',').add(1)

m_columns = ['is_new', 'rate', 'cost', 'n_cuisines', 'online_order', 'book_table']
for mc in m_columns:
    print(mc, ': ' , core[mc].dtype, 'NULL Count: ', core[mc].isna().sum())

core['is_new'].sum()
# n_cuisines - NaN means that the cuisine has not been listed yet hence it is unknown

is_new :  int64 NULL Count:  0
rate :  float64 NULL Count:  10052
cost :  float64 NULL Count:  346
n_cuisines :  float64 NULL Count:  45
online_order :  int64 NULL Count:  0
book_table :  int64 NULL Count:  0


np.int64(2208)

## Stage 3 - Standardize

In [3]:
# standardizing the text names - ['name', 'address', 'location']
t_names = ['name', 'address', 'location']
print(core.groupby(['name', 'address', 'location']).ngroups)
for c in t_names:
    core[c] = core[c].str.strip().str.replace(r'\s+', ' ', regex=True)
print(core.groupby(['name', 'address', 'location']).ngroups)

core['rate'].dropna().nunique()

12519
12519


31

In [4]:
print(core.groupby(['name', 'address']).ngroups)

12499


## Stage 4 - Validate

In [5]:
# rate must be a valid 5-point score - catches a parse that let 4.5/5 as 45
assert core['rate'].dropna().between(0, 5).all(), "rate out of [0, 5]"

# cost must be positive - catches a de-comma error that results in 0 or negative
assert (core['cost'].dropna() > 0).all(), "non-positive cost found"

# votes is a count - can be 0 but never negative
assert (core['votes'] >= 0).all(), "negative votes found"

# the keys that define our grain should never be null - a null key silently drops from every groupby
assert core[['name', 'address']].notna().all().all(), "Null key found"

print("All validation checks passed: ", len(core), " rows validated")

All validation checks passed:  51717  rows validated


## Stage 5 - Model to Schema

In [6]:
restaurants = core.groupby(['name', 'address']).agg(votes=('votes','max'), rate=('rate','mean'), cost=('cost','median'),
                                                    online_order=('online_order','max'), book_table=('book_table','max'),
                                                    listing_breadth=('listed_in(type)', 'nunique'),
                                                    location=('location','first'))

In [7]:
restaurants = restaurants.reset_index()
assert restaurants[['name', 'address']].duplicated().sum() == 0, "restraunt grain is not intact"
print("Number of rows: ", len(restaurants))
restaurants.head()

Number of rows:  12499


,name,address,votes,rate,cost,online_order,book_table,listing_breadth,location
0,#FeelTheROLL,"Opposite Mantri Commercio, Outer Ring Road, De...",7,3.4,200.0,0,0,1,Bellandur
1,#L-81 Cafe,"Sector 6, HSR Layout, HSR",48,3.9,400.0,1,0,2,HSR
2,#Vibes Restro,"Marasur Gate, Chandapura - Anekal Road, Near A...",0,NaN,700.0,0,0,3,Electronic City
3,#refuel,"7, Ground Floor, RR Commercial Complex, Akshay...",37,3.7,400.0,1,0,3,Bannerghatta Road
4,'Brahmins' Thatte Idli,"19, 1st main, 2nd cross, 3rd stage, 3rd block,...",0,NaN,100.0,1,0,1,Basaveshwara Nagar


In [8]:
# From the above snapshot of the restaurant table we can already see that there are cold-start restaurants
# with 0 votes and NULL ratings.
# Also there seems to be some correlation between the breadth of listings and the number of votes, though 
# a final conclusion can only be drawn upon further analysis.

In [9]:
listings = core[['name', 'address', 'listed_in(type)']].drop_duplicates().reset_index(drop=True)
assert listings.duplicated().sum() == 0, "listing grain not unique"
print("listing rows: ", len(listings))

listing rows:  20915


In [10]:
# now we have 20915 unique pairs of name, address and listings which means out of the total rows
# in the raw file about 59.6% were duplicate snapshot rows as a result of the multi-crawl scrape across
# time. 

In [11]:
chk_cuisine = core.groupby(['name', 'address'])['cuisines'].nunique()
print("Restaurants with non-unique cuisine values: ", (chk_cuisine > 1).sum())

Restaurants with non-unique cuisine values:  533


In [12]:
restaurant_cuisines = (core[['name', 'address', 'cuisines']]
                       .assign(cuisine=lambda d: d['cuisines'].str.split(','))
                       .explode('cuisine'))

In [13]:
restaurant_cuisines['cuisine'] = restaurant_cuisines['cuisine'].str.strip()

In [14]:
restaurant_cuisines = (restaurant_cuisines[['name', 'address', 'cuisine']]
                       .dropna(subset=['cuisine'])
                       .query("cuisine != ''")
                       .drop_duplicates()
                       .reset_index(drop=True))

# The above query acts as a union across different snapshots

In [15]:
assert restaurant_cuisines.duplicated().sum() == 0, "cuisine grain not unique"

print("cuisine rows: ", len(restaurant_cuisines),
      " | distinct: ", restaurant_cuisines['cuisine'].nunique(),
      " | per restaurant: ", round(len(restaurant_cuisines)/len(restaurants), 2))

cuisine rows:  29363  | distinct:  107  | per restaurant:  2.35


In [16]:
core['n_cuisines'].mean()

np.float64(2.4543079424059453)

In [17]:
n_cuis = restaurant_cuisines.groupby(['name', 'address']).size().rename('n_cuisines')
restaurants = restaurants.merge(n_cuis, on=['name', 'address'], how='left')

In [18]:
restaurants.columns

Index(['name', 'address', 'votes', 'rate', 'cost', 'online_order',
       'book_table', 'listing_breadth', 'location', 'n_cuisines'],
      dtype='str')

In [19]:
print(restaurants['n_cuisines'].mean())

2.3526159762839516


In [20]:
restaurants['n_cuisines'].isna().sum()

np.int64(18)

## Stage 6 - Persist

In [21]:
# Schema Contract - fail loudly rather than persist a broken table.

assert restaurants[['name', 'address']].duplicated().sum() == 0, "restaurant grain broken"
assert restaurant_cuisines.duplicated().sum() == 0, "Cuisine grain broken"
assert listings.duplicated().sum() == 0, "listing grain broken"
assert len(restaurants) == 12499, "restaurant count drifted"
assert restaurants['n_cuisines'].isna().sum() < 100, "unexpected cuisine nulls"

print("Schema contract passed succesfully")

Schema contract passed succesfully


In [22]:
restaurants.to_parquet('restaurants.parquet', index=False)
listings.to_parquet('listings.parquet', index=False)
restaurant_cuisines.to_parquet('restaurants_cuisines.parquet', index=False)

print("Persisted ->", len(restaurants),"restaurants |", len(listings), "listings |", len(restaurant_cuisines), "cuisine-links") 

Persisted -> 12499 restaurants | 20915 listings | 29363 cuisine-links
